# 02｜Window Attention：窗口划分与 QKV 形状

上一课已经知道，Swin Transformer 不让每个 token 立即查看整张图片，而是先把特征图划分成多个局部窗口，再在每个窗口内部计算 Self-Attention。

这一课只拆解普通 Window Attention，也就是 W-MSA。重点是跟踪窗口划分以及 Q、K、V 的形状，不讨论 Shifted Window 的移动和 mask。

## 1. 为什么要专门学习窗口划分

我们以前学习 Self-Attention 时，输入通常写成：

$$
X\in\mathbb{R}^{B\times N\times C}
$$

其中 $N$ 是序列长度。

Swin 的输入仍然来自二维图片，因此更自然的表示是：

$$
X\in\mathbb{R}^{B\times H\times W\times C}
$$

Window Attention 的第一项工作，就是把二维特征图重新分组，让每个局部窗口临时成为一条长度为 $M^2$ 的小序列。

这里没有删除 token，也没有改变 token 的特征内容，只是改变了它们的分组和排列方式。

![Window Attention 的窗口划分](images/02_window_partition.svg)

## 2. 本课使用的具体例子

以 Swin-T 的 Stage 1 为例：

| 符号 | 数值 | 含义 |
|---|---:|---|
| $B$ | 暂时保留 | batch 中的图片数量 |
| $H$ | 56 | 特征图高度 |
| $W$ | 56 | 特征图宽度 |
| $C$ | 96 | 每个 token 的特征维度 |
| $M$ | 7 | 窗口边长 |
| $h$ | 3 | 注意力头数 |

所以输入形状是：

$$
B\times56\times56\times96
$$

这里的 $56\times56$ 是 token 网格，不再是原始图片像素。每个 token 已经对应原图中的一个 $4\times4$ patch。

## 3. 第一步：计算窗口数量

窗口边长为 7，因此特征图的高度和宽度方向都能划分出：

$$
\frac{56}{7}=8\text{ 个窗口}
$$

每张图片的窗口总数为：

$$
n_w=\frac{H}{M}\times\frac{W}{M}=8\times8=64
$$

每个窗口内部包含：

$$
M^2=7\times7=49\text{ 个 tokens}
$$

可以检查 token 总数没有变化：

$$
64\times49=3136=56\times56
$$

窗口划分只是重新组织 3136 个 tokens，并没有减少或增加它们。

## 4. 第二步：把二维窗口变成小序列

为了看清分组过程，可以先把 $56\times56$ 拆成窗口网格和窗口内部网格：

$$
\begin{aligned}
B\times56\times56\times96
&\rightarrow B\times8\times7\times8\times7\times96 \\n&\rightarrow B\times8\times8\times7\times7\times96
\end{aligned}
$$

两个 8 表示纵向和横向的窗口数量，两个 7 表示一个窗口内部的高度和宽度。

接着把每个 $7\times7$ 窗口展平成长度为 49 的序列：

$$
B\times8\times8\times7\times7\times96
\rightarrow (B\cdot64)\times49\times96
$$

定义 $B_w=B\cdot64$，窗口序列就可以简写为：

$$
X_{\mathrm{win}}\in\mathbb{R}^{B_w\times49\times96}
$$

$B_w$ 不是新的图片数量，而是把 batch 中所有图片的窗口临时放到同一个批次维度中，方便并行计算。

## 5. 一个窗口就是一条局部 token 序列

窗口划分以后，每个窗口都可以独立看成一条序列：

$$
49\text{ 个 tokens}\times96\text{ 维特征}
$$

这与以前学习的序列 Self-Attention 完全对应：

- 序列长度 $N$ 由整张图片的 3136 变成单个窗口的 49；
- 特征维度 $C$ 仍然是 96；
- 不同窗口之间暂时互不计算 Attention；
- 所有窗口共享同一套 QKV 投影参数。

所以 Window Attention 不是一种全新的 Attention 公式，而是先改变 Self-Attention 的输入范围。

## 6. 第三步：生成 Q、K、V

窗口序列的每个 token 都要分别生成 Query、Key 和 Value：

$$
Q=X_{\mathrm{win}}W_Q,\qquad
K=X_{\mathrm{win}}W_K,\qquad
V=X_{\mathrm{win}}W_V
$$

三组投影都保持特征维度 96，因此：

$$
Q,K,V\in\mathbb{R}^{B_w\times49\times96}
$$

实际模型常把三次投影合并成一次从 96 维到 288 维的线性映射：

$$
B_w\times49\times96
\rightarrow B_w\times49\times(3\cdot96)
$$

再把最后的 288 维拆成 Q、K、V 三部分。这只是计算上的合并，数学含义仍然是三组不同的投影。

## 7. 第四步：拆成多个注意力头

Swin-T 的 Stage 1 使用 3 个注意力头，总特征维度为 96，因此每个 head 的维度是：

$$
d_{\mathrm{head}}=\frac{C}{h}=\frac{96}{3}=32
$$

Q、K、V 的形状分别从：

$$
B_w\times49\times96
$$

变为：

$$
B_w\times3\times49\times32
$$

四个维度依次表示：窗口批次、注意力头、窗口内 token 数量、每个 head 的特征维度。

3 个 head 并没有把总维度扩大到 3 倍，而是把原来的 96 维分成 3 份，每份 32 维。

## 8. 第五步：计算窗口内注意力分数

对每个窗口、每个 head，使用 Q 与 K 的转置相乘：

$$
(49\times32)\cdot(32\times49)
\rightarrow49\times49
$$

把窗口批次和注意力头维度加回来，注意力分数的完整形状是：

$$
QK^{\top}\in\mathbb{R}^{B_w\times3\times49\times49}
$$

最后两个 49 的含义不同：

- 倒数第二个 49 表示正在发起查询的 token；
- 最后一个 49 表示这个 token 可以查看的候选 tokens。

因此，每一行都描述窗口中某个 token 对 49 个窗口内 tokens 的关注分数。

## 9. 缩放、Softmax 与加权汇总

与普通 Self-Attention 一样，注意力分数先除以每个 head 维度的平方根：

$$
S=\frac{QK^{\top}}{\sqrt{d_{\mathrm{head}}}}
=\frac{QK^{\top}}{\sqrt{32}}
$$

然后沿最后一个维度使用 Softmax：

$$
A=\operatorname{softmax}(S)
$$

形状保持为：

$$
A\in\mathbb{R}^{B_w\times3\times49\times49}
$$

最后使用注意力权重对 V 加权汇总：

$$
(49\times49)\cdot(49\times32)
\rightarrow49\times32
$$

每个 token 因此得到一份融合了同窗口信息的新表示。

## 10. 第六步：合并多头并还原窗口

三个注意力头的输出形状是：

$$
B_w\times3\times49\times32
$$

把 3 个 head 重新拼接，得到：

$$
B_w\times49\times96
$$

经过输出投影后形状不变。接着把每个窗口的 49 个 tokens 恢复成 $7\times7$，再把 $8\times8$ 个窗口放回原位置：

$$
\begin{aligned}
(B\cdot64)\times49\times96
&\rightarrow B\times8\times8\times7\times7\times96 \\n&\rightarrow B\times56\times56\times96
\end{aligned}
$$

Window Attention 前后形状完全相同，因此它可以继续参与残差连接。变化的是每个 token 的内容，而不是 token 网格大小。

## 11. 完整 shape 路线

把前面的步骤串起来：

$$
\begin{aligned}
B\times56\times56\times96
&\xrightarrow{\text{划分窗口}} (B\cdot64)\times49\times96 \\n&\xrightarrow{\text{QKV 投影}} (B\cdot64)\times49\times288 \\n&\xrightarrow{\text{拆分 Q、K、V}} 3\times(B\cdot64)\times3\times49\times32 \\n&\xrightarrow{QK^{\top}} (B\cdot64)\times3\times49\times49 \\n&\xrightarrow{\text{加权汇总 V}} (B\cdot64)\times3\times49\times32 \\n&\xrightarrow{\text{合并多头}} (B\cdot64)\times49\times96 \\n&\xrightarrow{\text{还原窗口}} B\times56\times56\times96
\end{aligned}
$$

其中 QKV 拆分后的最前面那个 3 表示 Q、K、V 三组张量；紧接着的另一个 3 才表示三个注意力头。二者含义不同。

## 12. Window Attention 与全局 Attention 的真正区别

两者使用的核心公式没有改变，区别在于序列长度代表什么。

| 项目 | 全局 Attention | Window Attention |
|---|---:|---:|
| 一次 Attention 覆盖的 tokens | 整张特征图的 3136 个 | 单个窗口的 49 个 |
| 单个 head 的关系矩阵 | $3136\times3136$ | $49\times49$ |
| 是否立即跨窗口通信 | 是 | 否 |
| QKV 参数是否在窗口间共享 | 不涉及窗口 | 是 |

Window Attention 节省计算的关键，不是减少总 token 数，也不是降低每个 token 的维度，而是缩短每次 Self-Attention 实际处理的序列。

## 13. 本节小结

这一课需要掌握下面五点：

1. 窗口划分只重新组织 tokens，不改变 token 总数。
2. $56\times56$ 网格使用 $7\times7$ 窗口时，每张图片产生 64 个窗口，每个窗口有 49 个 tokens。
3. 每个窗口被看成一条 $49\times96$ 的局部序列。
4. Stage 1 使用 3 个注意力头，每个 head 的维度为 32。
5. 每个 head 的注意力矩阵只有 $49\times49$，输出还原后仍是 $56\times56\times96$。

Window Attention 解决了局部高效计算的问题，但普通窗口之间仍然无法通信。下一课再学习 Shifted Window 如何改变窗口分组并建立跨窗口联系。

## 14. 自测问题

1. 为什么 Swin 的输入常写成 $B\times H\times W\times C$？
2. $56\times56$ 网格使用 $7\times7$ 窗口时，每张图片有多少个窗口？
3. 为什么窗口划分后可以把形状写成 $(B\cdot64)\times49\times96$？
4. $B_w$ 表示真实图片数量吗？
5. QKV 合并投影为什么输出 288 维？
6. Stage 1 的每个 head 为什么是 32 维？
7. $QK^{\top}$ 为什么得到 $49\times49$，而不是 $96\times96$？
8. 注意力矩阵中两个 49 分别表示什么？
9. 多头输出合并后为什么重新变成 96 维？
10. Window Attention 前后形状为什么可以保持一致？
11. Window Attention 和全局 Attention 的公式是否完全不同？
12. 普通 Window Attention 还没有解决什么问题？

### 自测参考答案

1. Swin 保留 token 的二维空间网格，并把通道放在最后一个维度。
2. 每边 8 个，共 64 个窗口。
3. 每张图片的 64 个窗口被临时合并到批次维度，每个窗口展平为 49 个 96 维 tokens。
4. 不是。它表示 batch 中全部窗口组成的临时窗口批次，$B_w=B\cdot64$。
5. Q、K、V 各需要 96 维，合并后为 $3\times96=288$。
6. 总维度 96 被 3 个 head 平分，所以 $96/3=32$。
7. 矩阵乘法发生在 token 维度上：$(49\times32)\cdot(32\times49)$。
8. 前一个表示查询 token，后一个表示它可以查看的候选 token。
9. 三个 head 各输出 32 维，拼接后为 $3\times32=96$。
10. 窗口划分和窗口还原互为逆过程，Attention 只更新特征内容。
11. 不是。QKV、缩放点积、Softmax 和加权求和都相同，主要区别是作用范围。
12. 不同固定窗口之间仍然无法直接交换信息。